In [0]:
storage_account_name = "amazondatalake60308963"
storage_account_key = "YOUR_STORAGE_ACCOUNT_KEY"

spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
  storage_account_key)
print("ADLS access configured")

ADLS access configured


In [0]:
# Base paths for each container
raw_path       = f"abfss://raw@amazondatalake60308963.dfs.core.windows.net/cmapss"
processed_path = f"abfss://processed@amazondatalake60308963.dfs.core.windows.net/cmapss"
curated_path   = f"abfss://curated@amazondatalake60308963.dfs.core.windows.net/cmapss"

# List files in the raw container
dbutils.fs.ls(raw_path)

[FileInfo(path='abfss://raw@amazondatalake60308963.dfs.core.windows.net/cmapss/RUL_FD001.txt', name='RUL_FD001.txt', size=429, modificationTime=1773339623000),
 FileInfo(path='abfss://raw@amazondatalake60308963.dfs.core.windows.net/cmapss/test_FD001.txt', name='test_FD001.txt', size=2228855, modificationTime=1773339624000),
 FileInfo(path='abfss://raw@amazondatalake60308963.dfs.core.windows.net/cmapss/train_FD001.txt', name='train_FD001.txt', size=3515356, modificationTime=1773339624000)]

In [0]:
from pyspark.sql.functions import col

col_names = ['unit', 'cycle'] + [f'op_{i}' for i in range(1,4)] + [f'sensor_{i}' for i in range(1,22)]

# Load
train = spark.read.text(f"{raw_path}/train_FD001.txt")
test  = spark.read.text(f"{raw_path}/test_FD001.txt")
rul   = spark.read.text(f"{raw_path}/RUL_FD001.txt")

# Parse space-separated rows into columns
from pyspark.sql.functions import split, trim

def parse_txt(df, col_names):
    split_col = split(trim(df['value']), r'\s+')
    for i, name in enumerate(col_names):
        df = df.withColumn(name, split_col[i].cast('double'))
    return df.drop('value')

train = parse_txt(train, col_names)
test  = parse_txt(test,  col_names)
rul   = parse_txt(rul,   ['RUL'])

train.printSchema()
print(f"Train rows: {train.count()}")

root
 |-- unit: double (nullable = true)
 |-- cycle: double (nullable = true)
 |-- op_1: double (nullable = true)
 |-- op_2: double (nullable = true)
 |-- op_3: double (nullable = true)
 |-- sensor_1: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_5: double (nullable = true)
 |-- sensor_6: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_10: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor_13: double (nullable = true)
 |-- sensor_14: double (nullable = true)
 |-- sensor_15: double (nullable = true)
 |-- sensor_16: double (nullable = true)
 |-- sensor_17: double (nullable = true)
 |-- sensor_18: double (nullable = true)
 |-- sensor_19: double (nullable = true)
 |-- sensor_20: double (nullable = true)
 |-

In [0]:
from pyspark.sql.functions import max as spark_max

# Compute max cycle per engine then subtract current cycle
max_cycles = train.groupBy('unit').agg(spark_max('cycle').alias('max_cycle'))
train = train.join(max_cycles, on='unit')
train = train.withColumn('RUL', col('max_cycle') - col('cycle')).drop('max_cycle')

# Drop nulls
train = train.dropna()
test  = test.dropna()

train.select('unit', 'cycle', 'RUL').show(10)

+----+-----+-----+
|unit|cycle|  RUL|
+----+-----+-----+
| 8.0|  1.0|149.0|
| 8.0|  2.0|148.0|
| 8.0|  3.0|147.0|
| 8.0|  4.0|146.0|
| 8.0|  5.0|145.0|
| 8.0|  6.0|144.0|
| 8.0|  7.0|143.0|
| 8.0|  8.0|142.0|
| 8.0|  9.0|141.0|
| 8.0| 10.0|140.0|
+----+-----+-----+
only showing top 10 rows


In [0]:
print(f"Train rows: {train.count()}")
print(f"Train cols: {len(train.columns)}")
print(f"Test rows:  {test.count()}")
print(f"RUL rows:   {rul.count()}")

Train rows: 20631
Train cols: 27
Test rows:  13096
RUL rows:   100


In [0]:
from pyspark.sql.functions import stddev

sensor_cols = [f'sensor_{i}' for i in range(1,22)]

# Calculate std for each sensor
std_vals = train.select([stddev(c).alias(c) for c in sensor_cols]).collect()[0]

# Keep only sensors with std > 0.01
good_sensors = [s for s in sensor_cols if std_vals[s] and std_vals[s] > 0.01]
dropped = set(sensor_cols) - set(good_sensors)

print(f"Dropped: {dropped}")
print(f"Keeping {len(good_sensors)} sensors: {good_sensors}")

Dropped: {'sensor_6', 'sensor_16', 'sensor_18', 'sensor_1', 'sensor_19', 'sensor_10', 'sensor_5'}
Keeping 14 sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [0]:
from pyspark.ml.feature import MinMaxScaler as SparkMinMaxScaler
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# Assemble sensors into a vector
assembler = VectorAssembler(inputCols=good_sensors, outputCol='features_raw')
scaler = SparkMinMaxScaler(inputCol='features_raw', outputCol='features_scaled')

pipeline = Pipeline(stages=[assembler, scaler])
scaler_model = pipeline.fit(train)

train_scaled = scaler_model.transform(train)
test_scaled  = scaler_model.transform(test)

train_scaled.select('unit', 'cycle', 'RUL', 'features_scaled').show(3)

+----+-----+-----+--------------------+
|unit|cycle|  RUL|     features_scaled|
+----+-----+-----+--------------------+
|49.0|  1.0|214.0|[0.33132530120479...|
|49.0|  2.0|213.0|[0.34939759036144...|
|49.0|  3.0|212.0|[0.56024096385543...|
+----+-----+-----+--------------------+
only showing top 3 rows


In [0]:
from pyspark.ml.functions import vector_to_array

# Unpack vector back to individual columns
train_final = train_scaled.withColumn('scaled_array', vector_to_array('features_scaled'))
test_final  = test_scaled.withColumn('scaled_array',  vector_to_array('features_scaled'))

for i, s in enumerate(good_sensors):
    train_final = train_final.withColumn(s, col('scaled_array')[i])
    test_final  = test_final.withColumn(s,  col('scaled_array')[i])

# Keep only what we need
keep_cols = ['unit', 'cycle', 'RUL'] + good_sensors
train_final = train_final.select(keep_cols)
test_final  = test_final.select([c for c in keep_cols if c != 'RUL'])

# Save to processed container
train_final.write.mode('overwrite').parquet(f"{processed_path}/train_preprocessed.parquet")
test_final.write.mode('overwrite').parquet(f"{processed_path}/test_preprocessed.parquet")

print("Saved to processed container ✓")
train_final.show(3)

Saved to processed container ✓
+----+-----+-----+------------------+-------------------+-------------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+
|unit|cycle|  RUL|          sensor_2|           sensor_3|           sensor_4|          sensor_7|           sensor_8|           sensor_9|          sensor_11|         sensor_12|         sensor_13|          sensor_14|          sensor_15|          sensor_17|         sensor_20|         sensor_21|
+----+-----+-----+------------------+-------------------+-------------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+
| 8.0|  1.0|149.0| 0.593373493975889|  0.308044473512101| 0.4642133693450371|0.39613526570

In [0]:
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
import time

# Convert to Pandas
train_pd = train_final.toPandas()

ts_input = train_pd[['unit', 'cycle'] + good_sensors]

start = time.time()

extracted = extract_features(
    ts_input,
    column_id='unit',
    column_sort='cycle',
    n_jobs=4,             
    disable_progressbar=False
)

impute(extracted)

end = time.time()
print(f"Extraction done in {round(end - start, 2)}s")
print(f"Extracted features shape: {extracted.shape}")

Feature Extraction: 100%|██████████| 20/20 [01:48<00:00,  5.41s/it]
/local_disk0/.ephemeral_nfs/envs/pythonEnv-6782706f-a878-48a6-963a-1964f9bc7639/lib/python3.12/site-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['sensor_2__query_similarity_count__query_None__threshold_0.0'
 'sensor_3__query_similarity_count__query_None__threshold_0.0'
 'sensor_4__query_similarity_count__query_None__threshold_0.0'
 'sensor_7__query_similarity_count__query_None__threshold_0.0'
 'sensor_8__friedrich_coefficients__coeff_0__m_3__r_30'
 'sensor_8__friedrich_coefficients__coeff_1__m_3__r_30'
 'sensor_8__friedrich_coefficients__coeff_2__m_3__r_30'
 'sensor_8__friedrich_coefficients__coeff_3__m_3__r_30'
 'sensor_8__max_langevin_fixed_point__m_3__r_30'
 'sensor_8__query_similarity_count__query_None__threshold_0.0'
 'sensor_9__query_similarity_count__query_None__threshold_0.0'
 'sensor_11__query_similarity_count__query_None__threshold_0.0'
 'sensor_12__query_similarity_coun

Extraction done in 111.16s
Extracted features shape: (100, 10962)


In [0]:
import numpy as np
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
import pandas as pd

# RUL per engine
y = train_pd.groupby('unit')['RUL'].max()
y = y.loc[extracted.index]

# --- Pass 1: Variance threshold ---
vt = VarianceThreshold(threshold=0.01)
vt.fit(extracted)
extracted_vt = extracted.loc[:, vt.get_support()]
print(f"After variance filter: {extracted_vt.shape[1]} features")

# --- Pass 2: Pearson correlation with RUL (very fast) ---
pearson = extracted_vt.corrwith(y).abs()
extracted_pearson = extracted_vt[pearson.nlargest(150).index]
print(f"After Pearson filter: {extracted_pearson.shape[1]} features")

# --- Pass 3: MI on 150 features (fast now) ---
mi_scores = mutual_info_regression(extracted_pearson, y, random_state=42)
mi_series = pd.Series(mi_scores, index=extracted_pearson.columns).sort_values(ascending=False)
extracted_final = extracted_pearson[mi_series.head(50).index.tolist()]
print(f"After MI filter: {extracted_final.shape[1]} features")

After variance filter: 6701 features
After Pearson filter: 150 features
After MI filter: 50 features


In [0]:
# Save extracted_final to curated container
spark.createDataFrame(extracted_final.reset_index()).write.mode('overwrite').parquet(
    f"{curated_path}/train_features.parquet"
)
print(f"Saved {extracted_final.shape[1]} features to curated container ✓")

Saved 50 features to curated container ✓


In [0]:
%pip install deap

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
%pip install deap xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 21.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
extracted_final = spark.read.parquet(f"{curated_path}/train_features.parquet").toPandas()
print(extracted_final.columns[:5].tolist())
print(extracted_final.shape)

['index', 'sensor_9__range_count__max_1000000000000.0__min_0', 'sensor_15__range_count__max_1000000000000.0__min_0', 'sensor_8__length', 'sensor_7__range_count__max_1000000000000.0__min_0']
(100, 51)


In [0]:
extracted_final = spark.read.parquet(f"{curated_path}/train_features.parquet").toPandas()
extracted_final = extracted_final.set_index('index')

# Load train_pd for RUL labels
train_pd = spark.read.parquet(f"{processed_path}/train_preprocessed.parquet").toPandas()
y = train_pd.groupby('unit')['RUL'].max()
y = y.loc[extracted_final.index]

print(f"Features: {extracted_final.shape}")
print(f"RUL: {y.shape}")

Features: (100, 50)
RUL: (100,)


In [0]:

from deap import base, creator, tools, algorithms
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

X = extracted_final.values
feature_names = extracted_final.columns.tolist()
n_features = X.shape[1]  # 50

# --- GA Setup ---
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# --- Fitness Function ---
def evaluate(individual):
    selected = [i for i, bit in enumerate(individual) if bit == 1]
    
    # If no features selected, return worst score
    if len(selected) == 0:
        return (9999,)
    
    X_sel = X[:, selected]
    
    model = XGBRegressor(n_estimators=50, random_state=42, verbosity=0)
    rmse = np.sqrt(-cross_val_score(model, X_sel, y, cv=3, 
                                     scoring='neg_mean_squared_error').mean())
    
    # Penalize using too many features
    penalty = 0.1 * len(selected)
    
    return (rmse + penalty,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate",    tools.cxTwoPoint)
toolbox.register("mutate",  tools.mutFlipBit, indpb=0.05)
toolbox.register("select",  tools.selTournament, tournsize=3)

# --- Run GA ---
import time
start = time.time()

random.seed(42)
pop = toolbox.population(n=20)        # 20 individuals
result, log = algorithms.eaSimple(
    pop, toolbox,
    cxpb=0.7,       # crossover probability
    mutpb=0.2,      # mutation probability
    ngen=10,        # 10 generations
    verbose=True
)

end = time.time()
print(f"\nGA done in {round(end - start, 2)}s")

# --- Best individual ---
best = tools.selBest(result, k=1)[0]
selected_indices = [i for i, bit in enumerate(best) if bit == 1]
selected_features = [feature_names[i] for i in selected_indices]

print(f"Selected {len(selected_features)} features")

gen	nevals
0  	20    
1  	16    
2  	12    
3  	12    
4  	13    
5  	17    
6  	13    
7  	16    
8  	15    
9  	11    
10 	18    

GA done in 21.64s
Selected 10 features


In [0]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Get selected features
X_selected = extracted_final[selected_features].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

# Train
model = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = np.mean(np.abs(y_test - y_pred))
r2   = model.score(X_test, y_test)

print(f"RMSE: {round(rmse, 3)}")
print(f"MAE:  {round(mae, 3)}")
print(f"R2:   {round(r2, 3)}")
print(f"Features used: {len(selected_features)}")

RMSE: 4.802
MAE:  1.937
R2:   0.992
Features used: 10


In [0]:
#pipeline summary

print("=" * 40)
print("PIPELINE SUMMARY")
print("=" * 40)
print(f"Features after tsfresh:       10962")
print(f"Features after variance:       6701")
print(f"Features after Pearson:         150")
print(f"Features after MI:               50")
print(f"Features after GA:               10")
print("=" * 40)
print(f"tsfresh extraction time:  111.16s")
print(f"GA time:                   21.64s")
print("=" * 40)
print(f"RMSE:  4.802")
print(f"MAE:   1.937")
print(f"R2:    0.992")
print("=" * 40)

PIPELINE SUMMARY
Features after tsfresh:       10962
Features after variance:       6701
Features after Pearson:         150
Features after MI:               50
Features after GA:               10
tsfresh extraction time:  111.16s
GA time:                   21.64s
RMSE:  4.802
MAE:   1.937
R2:    0.992
